In [ ]:
import pandas as pd
import glob, os
from tqdm.notebook import tqdm
import json
import plotly.express as px
import plotly.graph_objects as go
import geopandas as gpd
import numpy as np
import chart_studio
import chart_studio.plotly as py
import plotly.io as pio
import re, string, unicodedata
import nltk
from bs4 import BeautifulSoup
from nltk import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', -1)


In [ ]:
username = 'politicai' # your username
api_key = 'REDACTED' # your api key - go to profile > settings > regenerate key
chart_studio.tools.set_credentials_file(username=username, api_key=api_key)
# py.plot(fig_object, filename = 'gdp_per_cap', auto_open=True)
# pio.write_html(fig, file=’index.html’, auto_open=True)

In [ ]:
class cleantext():
    
    def __init__(self, text = "test"):
        self.text = text
        self.banned_words = ["lectura", "fácil"]
        
    def strip_html(self):
        soup = BeautifulSoup(self.text, "html.parser")
        self.text = soup.get_text()
        return self

    def remove_between_square_brackets(self):
        self.text = re.sub('\[[^]]*\]', '', self.text)
        return self

    def remove_numbers(self):
        self.text = re.sub('[-+]?[0-9]+', '', self.text)
        return self
    
    def get_words(self):
        self.words = nltk.word_tokenize(self.text)
        return self

    def to_lowercase(self):
        """Convert all characters to lowercase from list of tokenized words"""
        new_words = []
        for word in self.words:
            new_word = word.lower()
            new_words.append(new_word)
        self.words = new_words
        return self

    def remove_punctuation(self):
        """Remove punctuation from list of tokenized words"""
        new_words = []
        for word in self.words:
            new_word = re.sub(r'[^\w\s]', '', word)
            if new_word != '':
                new_words.append(new_word)
        self.words = new_words
        return self

    def remove_stopwords(self):
        """Remove stop words from list of tokenized words"""
        new_words = []
        for word in self.words:
            if word not in stopwords.words('spanish'):
                new_words.append(word)
        self.words = new_words
        return self

    def remove_customwords(self):
        """Remove stop words from list of tokenized words"""
        new_words = []
        for word in self.words:
            if word not in self.banned_words:
                new_words.append(word)
        self.words = new_words
        return self

    def stem_words(self):
        """Stem words in list of tokenized words"""
        stemmer = SnowballStemmer('spanish')
        stems = []
        for word in self.words:
            stem = stemmer.stem(word)
            stems.append(stem)
        self.words = stems
        return self

    def lemmatize_verbs(self):
        """Lemmatize verbs in list of tokenized words"""
        lemmatizer = WordNetLemmatizer()
        lemmas = []
        for word in self.words:
            lemma = lemmatizer.lemmatize(word, pos='v')
            lemmas.append(lemma)
        self.words = lemmas
        return self
    
    def join_words(self):
        self.words = ' '.join(self.words)
        return self
    
    def do_all(self, text):
        self.text = text
        self = self.strip_html()
        self = self.remove_between_square_brackets()
        self = self.remove_numbers()        
        self = self.get_words()
        self = self.to_lowercase()
        self = self.remove_punctuation()
        self = self.remove_stopwords()
        self = self.remove_customwords()        
#        self = self.stem_words()
#        self = self.lemmatize_verbs()
        self = self.join_words()
        return self.words
    
    def do_all_stem(self, text):
        self.text = text
        self = self.strip_html()
        self = self.remove_between_square_brackets()
        self = self.remove_numbers()        
        self = self.get_words()
        self = self.to_lowercase()
        self = self.remove_punctuation()
        self = self.remove_stopwords()
        self = self.remove_customwords()        
        self = self.stem_words()
        self = self.lemmatize_verbs()
        self = self.join_words()
        return self.words    
    
# Define a function to plot word cloud
def plot_cloud(wordcloud):
    # Set figure size
    plt.figure(figsize=(40, 30))
    # Display image
    plt.imshow(wordcloud) 
    # No axis details
    plt.axis("off")
    
def wordListToFreqDict(wordlist):
    wordfreq = [wordlist.count(p) for p in wordlist]
    return dict(list(zip(wordlist,wordfreq)))

def cobertura_tema(texto, tema):
    perc = 100 * sum([1 for x in texto.split(" ") if x in tema]) / len(texto.split(" "))
    perc = np.round(perc,2)
    return perc

def cobertura_todos_temas(texto):
    cobertura_paro = cobertura_tema(texto, stem_paro)
    cobertura_politicos = cobertura_tema(texto, stem_politicos)
    cobertura_economia = cobertura_tema(texto, stem_economia)
    cobertura_cataluña = cobertura_tema(texto, stem_cataluña)
    cobertura_corrupcion = cobertura_tema(texto, stem_corrupcion)
    cobertura_sanidad = cobertura_tema(texto, stem_sanidad)
    dict_coberturas = {"paro":cobertura_paro, "políticos":cobertura_politicos, "economía":cobertura_economia, 
                        "cataluña":cobertura_cataluña, "corrupción":cobertura_corrupcion, "sanidad":cobertura_sanidad}
    return dict_coberturas

# Procesar PDFs

In [ ]:
txt_ciudadanos = open("../../data/staging/programas_electorales/201910/ciudadanos.txt", "r").read()
txt_podemos = open("../../data/staging/programas_electorales/201910/podemos.txt", "r").read()
txt_psoe = open("../../data/staging/programas_electorales/201910/psoe.txt", "r").read()
txt_mas_pais = open("../../data/staging/programas_electorales/201910/mas_pais.txt", "r").read()
txt_pp = open("../../data/staging/programas_electorales/201910/pp.txt", "r").read()
txt_vox = open("../../data/staging/programas_electorales/201910/vox.txt", "r").read()

In [ ]:
ct = cleantext()

In [ ]:
str_ciudadanos = ct.do_all(txt_ciudadanos)
str_podemos = ct.do_all(txt_podemos)
str_psoe = ct.do_all(txt_psoe)
str_mas_pais = ct.do_all(txt_mas_pais)
str_pp = ct.do_all(txt_pp)
str_vox = ct.do_all(txt_vox)

# Wordclouds

In [ ]:
freq_ciudadanos = wordListToFreqDict(str_ciudadanos.split(" "))
freq_podemos = wordListToFreqDict(str_podemos.split(" "))
freq_psoe = wordListToFreqDict(str_psoe.split(" "))
freq_mas_pais = wordListToFreqDict(str_mas_pais.split(" "))
freq_pp = wordListToFreqDict(str_pp.split(" "))
freq_vox = wordListToFreqDict(str_vox.split(" "))

In [ ]:
# Ciudadanos
wordcloud = WordCloud(width = 800, height = 400, random_state=1, background_color='darkorange', max_words=75, 
                      colormap='Pastel1', ranks_only=True, collocations=True, stopwords = STOPWORDS, 
                      normalize_plurals=True, min_word_length=3).generate_from_frequencies(freq_ciudadanos)
# Plot
plot_cloud(wordcloud)

In [ ]:
# Podemos
wordcloud = WordCloud(width = 800, height = 400, random_state=1, background_color='mediumpurple', max_words=75, 
                      colormap='Pastel1', ranks_only=True, collocations=True, stopwords = STOPWORDS, 
                      normalize_plurals=True, min_word_length=3).generate_from_frequencies(freq_podemos)
# Plot
plot_cloud(wordcloud)

In [ ]:
# PSOE
wordcloud = WordCloud(width = 800, height = 400, random_state=1, background_color='lightcoral', max_words=75, 
                      colormap='Pastel1', ranks_only=True, collocations=True, stopwords = STOPWORDS, 
                      normalize_plurals=True, min_word_length=3).generate_from_frequencies(freq_psoe)
# Plot
plot_cloud(wordcloud)

In [ ]:
# Más País
wordcloud = WordCloud(width = 800, height = 400, random_state=1, background_color='aquamarine', max_words=75, 
                      colormap='tab20b', ranks_only=True, collocations=True, stopwords = STOPWORDS, 
                      normalize_plurals=True, min_word_length=3).generate_from_frequencies(freq_mas_pais)
# Plot
plot_cloud(wordcloud)

In [ ]:
# PP
wordcloud = WordCloud(width = 800, height = 400, random_state=1, background_color='cornflowerblue', max_words=75, 
                      colormap='Pastel1', ranks_only=True, collocations=True, stopwords = STOPWORDS, 
                      normalize_plurals=True, min_word_length=3).generate_from_frequencies(freq_pp)
# Plot
plot_cloud(wordcloud)

In [ ]:
# Vox
wordcloud = WordCloud(width = 800, height = 400, random_state=1, background_color='limegreen', max_words=75, 
                      colormap='Pastel1', ranks_only=True, collocations=True, stopwords = STOPWORDS, 
                      normalize_plurals=True, min_word_length=3).generate_from_frequencies(freq_vox)
# Plot
plot_cloud(wordcloud)

# Principales problemas

In [ ]:
df_cis = pd.read_excel("../../data/staging/cis_nov19.xlsx",header=0)
df_cis = df_cis.sort_values(by="Porcentaje", ascending=False)
df_cis.head(6)

In [ ]:
palabras_paro = "paro prestación erte ere desempleo inem trabajo trabajar trabajador pyme autonomo empleo desocupacion huelga salario despido destitución inactividad cese subsidio".split(" ")
palabras_politicos = "político política partido corrupción diputados aforado".split(" ")
palabras_economia = "dinero economía salario empleo trabajo pensiones nómina desigualdad".split(" ")
palabras_cataluña = "cataluña procés nacionalismo independencia independentistas ruptura constitución president referéndum".split(" ")
palabras_corrupcion = "corrupción nepotismo giratorias enchufe robar juicio trama cárcel fraude sobresueldo".split(" ")
palabras_sanidad = "sanidad salud intervención vacuna cirugía hospital medico enfermero ambulatorio operación".split(" ")

In [ ]:
stem_paro = ct.do_all_stem(" ".join(palabras_paro))
stem_politicos = ct.do_all_stem(" ".join(palabras_politicos))
stem_economia = ct.do_all_stem(" ".join(palabras_economia))
stem_cataluña = ct.do_all_stem(" ".join(palabras_cataluña))
stem_corrupcion = ct.do_all_stem(" ".join(palabras_corrupcion))
stem_sanidad = ct.do_all_stem(" ".join(palabras_sanidad))

In [ ]:
cob_ciudadanos = cobertura_todos_temas(ct.do_all_stem(str_ciudadanos))
cob_podemos = cobertura_todos_temas(ct.do_all_stem(str_podemos))
cob_psoe = cobertura_todos_temas(ct.do_all_stem(str_psoe))
cob_mas_pais = cobertura_todos_temas(ct.do_all_stem(str_mas_pais))
cob_pp = cobertura_todos_temas(ct.do_all_stem(str_pp))
cob_vox = cobertura_todos_temas(ct.do_all_stem(str_vox))

In [ ]:
col_ciudadanos = pd.DataFrame.from_dict(cob_ciudadanos, orient="index").rename(columns={0:"Ciudadanos"})
col_podemos = pd.DataFrame.from_dict(cob_podemos, orient="index").rename(columns={0:"Podemos"})
col_psoe = pd.DataFrame.from_dict(cob_psoe, orient="index").rename(columns={0:"PSOE"})
col_mas_pais = pd.DataFrame.from_dict(cob_mas_pais, orient="index").rename(columns={0:"Más País"})
col_pp = pd.DataFrame.from_dict(cob_pp, orient="index").rename(columns={0:"PP"})
col_vox = pd.DataFrame.from_dict(cob_vox, orient="index").rename(columns={0:"Vox"})

col_ciudadanos["Podemos"] = col_podemos
col_ciudadanos["PSOE"] = col_psoe
col_ciudadanos["Más País"] = col_mas_pais
col_ciudadanos["PP"] = col_pp
col_ciudadanos["Vox"] = col_vox
df_cob = col_ciudadanos

In [ ]:
df_cob

In [ ]:
color_map = {'PSOE':"#E52B50", 
             'PP':"#1E90FF",  
             'Ciudadanos':"#FFA500",                
             'Podemos':"#9370DB",
             'Vox':"#00FF00",
              'Más País':"#20FFA1"}

In [ ]:
px.line(data_frame=df_cob, color_discrete_map=color_map, template="plotly_white",
       title= f"Cobertura de los problemas de los españoles (nov 19) por tema y partido político",
       labels={"value": "Porcentaje",
               "index": "Tema"})

In [ ]:
px.bar(data_frame=df_cob.T, template="plotly_white",
       title= f"Cobertura de los problemas de los españoles (nov 19) por partido político y tema",
       labels={"value": "Porcentaje","index": "Partido"}).update_xaxes(categoryorder="total descending")